# 16.3 Vector-space models

Vector-space retrieval represents queries and documents as weighted term vectors, then ranks by angle rather than exact Boolean membership.

Vector-space models soften the hard yes-or-no retrieval of 16.1 while keeping the sparse, inspectable vocabulary. Counts become coordinates in a document-term matrix, IDF rescales rare coordinates, and cosine similarity ranks by angle. Save a copy to Drive to edit.

In [ ]:

import math
import re
import signal
import random
from collections import Counter
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

SEED = 16
random.seed(SEED)
np.random.seed(SEED)


def tokenize(text, lowercase=True, synonym_map=None, keep_case=False):
    if lowercase and not keep_case:
        text = text.lower()
    tokens = re.findall(r"[A-Za-z0-9_'-]+", text)
    if synonym_map is None:
        return tokens
    mapped = []
    for token in tokens:
        key = token.lower()
        mapped.append(synonym_map.get(key, key))
    return mapped


def ranked_doc_ids(scores, reverse=True):
    return [doc_id for doc_id, score in sorted(scores.items(), key=lambda item: (-item[1], item[0]) if reverse else (item[1], item[0]))]


def recall_at_k(ranking, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    hits = len(set(ranking[:k]) & relevant)
    return hits / len(relevant)


def dcg_at_k(ranking, gains, k):
    total = 0.0
    for rank, doc_id in enumerate(ranking[:k], start=1):
        gain = gains.get(doc_id, 0.0)
        total += gain / math.log2(rank + 1)
    return total


def ndcg_at_k(ranking, gains, k):
    ideal = sorted(gains.values(), reverse=True)
    ideal_total = 0.0
    for rank, gain in enumerate(ideal[:k], start=1):
        ideal_total += gain / math.log2(rank + 1)
    if ideal_total == 0:
        return 0.0
    return dcg_at_k(ranking, gains, k) / ideal_total


def mrr(ranking, relevant):
    relevant = set(relevant)
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def preview_rungs(rungs):
    for rung in rungs:
        docs = rung["docs"]
        print(rung["name"], "docs=", len(docs), "query=", rung["query"])
        sample_ids = list(docs)[:3]
        for doc_id in sample_ids:
            print(" ", doc_id, "->", docs[doc_id][:90])
        print()


def try_fetch_20newsgroups_subset(categories, max_docs=18, timeout_seconds=3):
    def timeout_handler(signum, frame):
        raise TimeoutError("fetch_20newsgroups timed out")
    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout_seconds)
    try:
        from sklearn.datasets import fetch_20newsgroups
        data = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
        texts = [text.replace("\n", " ") for text in data.data[:max_docs]]
        return texts
    except Exception:
        return None
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


def inline_newsgroups_docs():
    return {
        1: "space shuttle orbit nasa launch telescope mission",
        2: "hockey puck goalie playoff ice team",
        3: "graphics rendering image pixel shader gpu",
        4: "nasa mars orbit satellite telemetry",
        5: "goalie blocks puck in hockey playoff",
        6: "image compression rendering color pixel",
        7: "space telescope observes galaxy orbit",
        8: "team wins ice hockey tournament",
        9: "gpu shader renders three dimensional graphics",
        10: "satellite launch vehicle reaches orbit",
        11: "playoff team trades veteran goalie",
        12: "graphics image pipeline uses antialiasing",
    }


def lesson_docs_boolean():
    return {
        1: "neural search search",
        2: "search graph",
        3: "neural retrieval",
        4: "graph retrieval search search",
    }


def lesson_docs_sets():
    return {
        1: "neural search",
        2: "search graph",
        3: "neural retrieval",
        4: "graph retrieval search",
    }


def retrieval_ladder(kind):
    if kind == "boolean":
        return [
            {
                "name": "D1 lesson toy corpus",
                "docs": lesson_docs_sets(),
                "query": "search AND graph",
                "relevant": {2, 4},
            },
            {
                "name": "D2 12 clean topic docs",
                "docs": {
                    1: "neural search ranking",
                    2: "graph retrieval index",
                    3: "recipe ingredient oven",
                    4: "neural embeddings search",
                    5: "legal discovery contract",
                    6: "graph database traversal",
                    7: "search engine crawler",
                    8: "retrieval evaluation recall",
                    9: "support ticket reset",
                    10: "neural network training",
                    11: "index postings compression",
                    12: "semantic search retrieval",
                },
                "query": "neural AND search",
                "relevant": {1, 4},
            },
            {
                "name": "D3 case synonyms ties",
                "docs": {
                    1: "Search platform handles neural queries",
                    2: "semantic matching helps search",
                    3: "NEURAL retrieval benchmark",
                    4: "graph search retrieval",
                    5: "support SEARCH portal",
                    6: "neural Search ranking",
                    7: "lexical lookup only",
                    8: "semantic retrieval without keyword",
                    9: "search neural tie case",
                    10: "faq finder synonym lookup",
                    11: "audit log exclusion filter",
                    12: "rareterm alpha beta",
                },
                "query": "neural AND search",
                "relevant": {1, 6, 9},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space AND orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 long-tail negation rare terms",
                "docs": {
                    1: "urgent security incident xj9 raretoken search neural allowlist",
                    2: "security incident xj9 raretoken malware neural",
                    3: "Search incident raretoken benign customer support",
                    4: "security search xj9 raretoken not neural",
                    5: "compliance archive common logs",
                    6: "raretoken investigation search neural case study",
                    7: "neural search raretoken exclude deprecated",
                    8: "xj9 raretoken Search security search",
                    9: "incident response without keyword",
                    10: "rareterm unrelated graph retrieval",
                    11: "security neural search raretoken",
                    12: "deprecated neural search raretoken",
                },
                "query": "raretoken AND search AND NOT deprecated",
                "relevant": {1, 3, 4, 6, 8, 11},
            },
        ]
    if kind == "bm25":
        return [
            {
                "name": "D1 lesson BM25 corpus",
                "docs": lesson_docs_boolean(),
                "query": "neural search",
                "gains": {1: 3, 3: 2, 4: 1, 2: 0},
            },
            {
                "name": "D2 clean keyword docs",
                "docs": {
                    1: "neural search ranking relevance",
                    2: "search ranking index",
                    3: "neural retrieval embeddings",
                    4: "graph database traversal",
                    5: "support search help center",
                    6: "neural network optimizer",
                    7: "enterprise search neural",
                    8: "retrieval evaluation ndcg",
                    9: "legal discovery query",
                    10: "search logs observability",
                    11: "neural search relevance",
                    12: "lexical index postings",
                },
                "query": "neural search",
                "gains": {1: 3, 7: 3, 11: 3, 3: 1, 6: 1, 2: 1, 5: 1, 10: 1},
            },
            {
                "name": "D3 stuffed repeats length variation",
                "docs": {
                    1: "neural search relevance",
                    2: "search search search search search coupon",
                    3: "neural retrieval scientific paper",
                    4: "search neural search neural ranking",
                    5: "very long document with search filler filler filler filler filler",
                    6: "graph traversal retrieval",
                    7: "neural semantic search",
                    8: "search support support support",
                    9: "neural neural model only",
                    10: "ranking index postings",
                    11: "search neural anti stuffing",
                    12: "irrelevant repeated repeated repeated",
                },
                "query": "neural search",
                "gains": {1: 3, 4: 3, 7: 3, 11: 3, 3: 1, 9: 1, 2: 0, 5: 0},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "gains": {1: 3, 4: 3, 7: 3, 10: 3},
            },
            {
                "name": "D5 long docs synonym misses",
                "docs": {
                    1: "neural search ranking tutorial with exact query terms",
                    2: "semantic retrieval finds meaning with embeddings and vectors",
                    3: "enterprise findability platform with semantic matching but no keyword overlap",
                    4: "neural neural neural search search stuffed landing page",
                    5: "long audit report search appendix appendix appendix appendix appendix appendix",
                    6: "support answer retrieval help article",
                    7: "semantic vector matching for neural information need",
                    8: "search logs with unrelated navigation data",
                    9: "dense encoder synonym recall for faq finder",
                    10: "legal discovery keyword exact match",
                    11: "neural search production benchmark",
                    12: "meaning based lookup with embeddings",
                },
                "query": "neural search",
                "gains": {1: 3, 2: 2, 3: 2, 7: 2, 9: 2, 11: 3, 4: 1},
            },
        ]
    if kind == "tfidf":
        return [
            {
                "name": "D1 lesson sparse vectors",
                "docs": lesson_docs_boolean(),
                "query": "neural search",
                "relevant": {1, 3, 4},
            },
            {
                "name": "D2 clean TF-IDF mini corpus",
                "docs": {
                    1: "neural search ranking",
                    2: "search index postings",
                    3: "neural embeddings retrieval",
                    4: "graph database",
                    5: "semantic search retrieval",
                    6: "support center",
                    7: "neural search evaluation",
                    8: "legal discovery",
                    9: "query expansion synonym",
                    10: "search logs",
                    11: "neural model",
                    12: "ranking relevance search",
                },
                "query": "neural search",
                "relevant": {1, 7, 3, 5, 11},
            },
            {
                "name": "D3 stopwords synonyms ties",
                "docs": {
                    1: "the neural search system",
                    2: "the car repair guide",
                    3: "automobile maintenance manual",
                    4: "search and the retrieval",
                    5: "neural retrieval and search",
                    6: "vehicle diagnostics handbook",
                    7: "the the the search",
                    8: "semantic lookup finder",
                    9: "neural model guide",
                    10: "the and of support",
                    11: "search neural duplicate",
                    12: "graph retrieval index",
                },
                "query": "neural search",
                "relevant": {1, 5, 9, 11},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 long-tail vocab stopword-heavy docs",
                "docs": {
                    1: "car insurance claim policy vehicle accident",
                    2: "automobile coverage deductible crash report",
                    3: "the the the and of to in car",
                    4: "vehicle repair parts estimate",
                    5: "neural search unrelated tutorial",
                    6: "automobile automobile warranty service",
                    7: "car rental booking airport",
                    8: "rare sku xj9 part compatibility vehicle",
                    9: "support article about login reset",
                    10: "auto policy renewal notice",
                    11: "vehicle crash legal claim",
                    12: "long stopword document the and of in to with for on by",
                },
                "query": "car accident claim",
                "relevant": {1, 2, 4, 8, 10, 11},
            },
        ]
    if kind == "dense":
        return [
            {
                "name": "D1 lesson 2D embeddings",
                "docs": {
                    1: "semantic search match",
                    2: "faq synonym neighbor",
                    3: "orthogonal mismatch",
                    4: "opposite constraint",
                },
                "query": "semantic search",
                "embeddings": np.array([[1.0, 0.0], [0.8, 0.2], [0.0, 1.0], [-0.2, 0.9]]),
                "query_vector": np.array([0.9, 0.1]),
                "relevant": {1, 2},
            },
            {
                "name": "D2 clean semantic clusters",
                "docs": {i + 1: text for i, text in enumerate([
                    "password reset login help",
                    "account sign in recovery",
                    "invoice billing payment",
                    "credit card receipt",
                    "model training embeddings",
                    "vector retrieval semantic",
                    "shipping delivery package",
                    "order tracking courier",
                    "legal contract review",
                    "compliance policy audit",
                    "search ranking relevance",
                    "faq answer support",
                ])},
                "query": "login recovery",
                "relevant": {1, 2, 12},
            },
            {
                "name": "D3 synonyms noisy high-norm vectors",
                "docs": {i + 1: text for i, text in enumerate([
                    "car repair manual",
                    "automobile service guide",
                    "vehicle maintenance tips",
                    "billing payment invoice",
                    "login account help",
                    "huge norm noisy unrelated",
                    "auto insurance claim",
                    "engine diagnostic checklist",
                    "travel hotel booking",
                    "graph neural search",
                    "support ticket reset",
                    "semantic finder retrieval",
                ])},
                "query": "car maintenance",
                "relevant": {1, 2, 3, 7, 8},
            },
            {
                "name": "D4 20newsgroups tiny hashed SVD embeddings",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 off-domain constraint-heavy queries",
                "docs": {i + 1: text for i, text in enumerate([
                    "refund policy for enterprise plan excluding trials",
                    "trial cancellation guide no refund guaranteed",
                    "enterprise contract renewal refund clause",
                    "sports hockey playoff recap",
                    "graphics gpu shader discussion",
                    "refund exception for medical emergency",
                    "login reset support article",
                    "enterprise billing dispute with legal constraint",
                    "semantic neighbor about repayment but missing enterprise",
                    "off domain recipe ingredients",
                    "legal contract refund negotiation",
                    "high norm generic vector placeholder",
                ])},
                "query": "enterprise refund not trial",
                "relevant": {1, 3, 8, 11},
            },
        ]
    raise ValueError(kind)


def tfidf_matrix(docs, synonym_map=None, stopwords=None):
    tokenized = {}
    for doc_id, text in docs.items():
        tokens = tokenize(text, synonym_map=synonym_map)
        if stopwords is not None:
            tokens = [token for token in tokens if token not in stopwords]
        tokenized[doc_id] = tokens
    vocab = sorted({token for tokens in tokenized.values() for token in tokens})
    doc_ids = list(docs)
    counts = np.zeros((len(doc_ids), len(vocab)))
    for row, doc_id in enumerate(doc_ids):
        counter = Counter(tokenized[doc_id])
        for col, term in enumerate(vocab):
            counts[row, col] = counter.get(term, 0)
    df = np.count_nonzero(counts > 0, axis=0)
    idf = np.log((len(doc_ids) + 1.0) / (df + 1.0)) + 1.0
    vectors = counts * idf
    return doc_ids, vocab, vectors, idf


def tfidf_cosine_rank(query, docs, synonym_map=None, stopwords=None):
    doc_ids, vocab, vectors, idf = tfidf_matrix(docs, synonym_map=synonym_map, stopwords=stopwords)
    query_tokens = tokenize(query, synonym_map=synonym_map)
    if stopwords is not None:
        query_tokens = [token for token in query_tokens if token not in stopwords]
    query_counter = Counter(query_tokens)
    query_vector = np.array([query_counter.get(term, 0) for term in vocab], dtype=float) * idf
    query_norm = np.linalg.norm(query_vector)
    scores = {}
    for row, doc_id in enumerate(doc_ids):
        doc_norm = np.linalg.norm(vectors[row])
        if query_norm == 0 or doc_norm == 0:
            scores[doc_id] = 0.0
        else:
            scores[doc_id] = float(np.dot(query_vector, vectors[row]) / (query_norm * doc_norm))
    return scores, doc_ids, vocab, vectors, idf, query_vector


## The concept, built once (D1)

With smoothed $idf_j=\log((N+1)/(df_j+1))+1$, a TF-IDF vector is $v_i=X_i\odot idf$. Query-document similarity is $$\cos(q,v_i)=rac{q^	op v_i}{\|q\|_2\|v_i\|_2}$$ for sparse vocabulary coordinates.

In [ ]:

docs = lesson_docs_boolean()
scores, doc_ids, vocab, vectors, idf, query_vector = tfidf_cosine_rank("neural search", docs)
idf_lookup = dict(zip(vocab, idf))
query_norm = float(np.linalg.norm(query_vector))

assert round(idf_lookup["neural"], 3) == 1.511
assert round(idf_lookup["search"], 3) == 1.223
assert round(query_norm, 3) == 1.944

print("vocab", vocab)
print("idf(neural)", round(idf_lookup["neural"], 3))
print("idf(search)", round(idf_lookup["search"], 3))
print("query norm", round(query_norm, 3))


The lesson's repeated `search` count tilts $d_1$ away from the exact query direction but keeps it first: $d_1=0.944>d_3=0.550>d_4=0.474>d_2=0.396$.

In [ ]:

ranking = ranked_doc_ids(scores)

assert round(scores[1], 3) == 0.944
assert round(scores[3], 3) == 0.550
assert round(scores[4], 3) == 0.474
assert round(scores[2], 3) == 0.396
assert ranking == [1, 3, 4, 2]

for doc_id in ranking:
    print(doc_id, round(scores[doc_id], 3))


## The dataset ladder

D1 is the lesson vocabulary, D2 is a clean TF-IDF mini corpus, D3 adds stopwords, synonyms, and ties, D4 is a three-category 20newsgroups-style corpus, and D5 stresses long-tail vocabulary plus stopword-heavy documents.

In [ ]:

rungs = retrieval_ladder("tfidf")
preview_rungs(rungs)


In [ ]:

rows = []
artifacts = []
for rung in rungs:
    scores, doc_ids, vocab, vectors, idf, query_vector = tfidf_cosine_rank(rung["query"], rung["docs"])
    ranking = ranked_doc_ids(scores)
    metric = mrr(ranking, rung["relevant"])
    rows.append((rung["name"], metric, ranking[:5]))
    artifacts.append((rung, scores, doc_ids, vocab, vectors, query_vector, ranking))

print("rung | MRR | top5")
for name, metric, top5 in rows:
    print(f"{name} | {metric:.3f} | {top5}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, (rung, scores, doc_ids, vocab, vectors, query_vector, ranking) in enumerate(artifacts):
    show = vectors[: min(6, vectors.shape[0]), : min(8, vectors.shape[1])]
    axes[0, col].imshow(show, aspect="auto")
    axes[0, col].set_title(rung["name"].split()[0])
    axes[0, col].set_xlabel("terms")
    top = ranking[:5]
    axes[1, col].bar([str(doc_id) for doc_id in top], [scores[doc_id] for doc_id in top])
    axes[1, col].set_ylim(0, 1.05)
    axes[1, col].set_title("cosine slate")

metric_values = [row[1] for row in rows]
fig2, ax = plt.subplots(figsize=(6, 3))
ax.plot(["D1", "D2", "D3", "D4", "D5"], metric_values, marker="o")
ax.set_ylim(0, 1.05)
ax.set_ylabel("MRR")
ax.set_title("MRR vs complexity")
plt.show()


## Pitfall on D5: sparse vectors miss synonyms

`car` and `automobile` are different sparse coordinates. A transparent synonym map can improve the hardest-rung ranking, while the caveat remains that real semantic matching needs learned representations.

In [ ]:

d5 = rungs[-1]
base_scores, base_doc_ids, base_vocab, base_vectors, base_idf, base_query = tfidf_cosine_rank(d5["query"], d5["docs"])
base_ranking = ranked_doc_ids(base_scores)
base_metric = mrr(base_ranking, d5["relevant"])
synonym_map = {
    "automobile": "car",
    "auto": "car",
    "vehicle": "car",
    "crash": "accident",
    "coverage": "claim",
    "policy": "claim",
}
fixed_scores, fixed_doc_ids, fixed_vocab, fixed_vectors, fixed_idf, fixed_query = tfidf_cosine_rank(d5["query"], d5["docs"], synonym_map=synonym_map)
fixed_ranking = ranked_doc_ids(fixed_scores)
fixed_metric = mrr(fixed_ranking, d5["relevant"])

assert fixed_metric >= base_metric
print("base top5", base_ranking[:5], "MRR", round(base_metric, 3))
print("synonym top5", fixed_ranking[:5], "MRR", round(fixed_metric, 3))


## Evaluate it + Practice

- Report the planned metric (MRR) beside a no-skill baseline such as random ranking or first-k document order.
- Run a cheap sanity check: the D1 asserts should pass before trusting any harder rung.
- Ablate the key idea and expect the metric to drop: replace cosine with raw dot product or remove synonym mapping on D5.
- Watch failure signals: empty candidates, tied scores, case splits, synonym misses, and high-norm artifacts.
- Keep all examples CPU-only, seeded, and small enough for a notebook cell.

Practice prompts:
1. Change one relevance label on D3 and recompute the metric table.



2. Add a stopword list to D3 and compare the heatmap.

3. Change the IDF smoothing convention and rerun the D1 asserts after updating expected values.